# ICM-UADE — Mapa de precios por sucursal
**INECO · Universidad Argentina de la Empresa**

Genera el mapa interactivo, rankings de cadenas, provincias y barrios CABA.

▶️ **Ejecutar todo** (`Runtime → Run all`) y esperar ~10 minutos.
La única celda modificable es la primera — para elegir el mes.

In [ ]:
# ══════════════════════════════════════════════════════════════════
# CONFIGURACIÓN — única celda que podés modificar
# ══════════════════════════════════════════════════════════════════

# Mes a analizar (formato 'YYYY-MM').
# Dejalo en None para usar el último mes disponible automáticamente.
MES = None   # ← cambiá a '2026-04' si querés un mes específico

# ── No modificar nada de acá para abajo ───────────────────────────
REPO_URL     = 'https://github.com/santiagoriverti/analisis_precios_minoristas_UADE.git'
MANIFEST_URL = 'https://raw.githubusercontent.com/santiagoriverti/analisis_precios_minoristas_UADE/main/data/drive_manifest.json'
DIR_DATOS    = '/content/datos_sepa'
DIR_REPO     = '/content/repo'

# IDs de las carpetas de Drive de INECO (no modificar)
FOLDER_HISTORICO = '13GONeBs5lQCSUdBioHYk-8GhfDtIyliD'  # 2018-2023
FOLDER_RECIENTE  = '1GNs9SrZ4BIoBsviBVWYYqRcsj4dwPF-I'  # 2024-presente

NOMBRE_MAESTRO_PROD = 'Maestro de Productos Interno.xlsx'
NOMBRE_MAESTRO_SUC  = 'maestro_sucursales_completo.xlsx'

In [ ]:
# ══════════════════════════════════════════════════════════════════
# PASO 1 — Instalación y clonado
# ══════════════════════════════════════════════════════════════════
import os, sys

if not os.path.exists(DIR_REPO):
    os.system(f'git clone -q {REPO_URL} {DIR_REPO}')
    print('✓ Repositorio clonado')
else:
    os.system(f'git -C {DIR_REPO} pull -q')
    print('✓ Repositorio actualizado')

os.system('pip install -q gdown folium branca openpyxl pyarrow')
sys.path.insert(0, f'{DIR_REPO}/src')
os.makedirs(DIR_DATOS, exist_ok=True)
print('✓ Listo')

In [ ]:
# ══════════════════════════════════════════════════════════════════
# PASO 2 — Detectar método de descarga
#
# Los datos en Drive están en ZIPs anuales/semestrales:
#   2026.zip, 2025.zip, 2024.zip, 2021A.zip, 2021B.zip, ...
# Cada ZIP contiene pares de CSV.GZ por mes:
#   MMAAAA_pais_parte1COMPLETO.csv.gz + parte2COMPLETO.csv.gz
#
# Intenta primero el manifest (sin auth → gdown descarga el ZIP).
# Si no hay IDs en el manifest, usa la API de Drive (auth una vez).
# ══════════════════════════════════════════════════════════════════
import requests, json, re

manifest = requests.get(MANIFEST_URL).json()

# Meses con IDs reales en el manifest (estructura nueva: zips con lista de meses)
zips_manifest = {
    nombre: info for nombre, info in manifest.get('zips', {}).items()
    if isinstance(info, dict)
    and info.get('id', 'COMPLETAR') not in ('COMPLETAR', '', None)
    and not nombre.startswith('_')
}

USAR_MANIFEST = len(zips_manifest) > 0

if USAR_MANIFEST:
    todos_meses_manifest = sorted(
        m for info in zips_manifest.values() for m in info.get('meses', [])
    )
    print(f'✓ Manifest cargado — {len(zips_manifest)} ZIPs disponibles')
    print(f'  Meses cubiertos: {todos_meses_manifest[0]} → {todos_meses_manifest[-1]}')

    MES_ANALIZAR = MES if (MES and MES in todos_meses_manifest) else todos_meses_manifest[-1]
    if MES and MES not in todos_meses_manifest:
        print(f'  ⚠ {MES} no está en el manifest — usando el último: {MES_ANALIZAR}')
    print(f'  Mes seleccionado: {MES_ANALIZAR}')
else:
    print('ℹ El manifest aún no tiene IDs de ZIPs.')
    print('  Se usará la API de Google Drive → vas a ver un popup de autorización.')
    MES_ANALIZAR = MES  # puede ser None, se resuelve en PASO 3

In [ ]:
# ══════════════════════════════════════════════════════════════════
# PASO 3 — Descargar y extraer archivos
#
# Flujo:
#   1. Identificar qué ZIP anual/semestral contiene el mes target
#   2. Descargar ese ZIP (gdown si hay manifest, Drive API si no)
#   3. Extraer solo los 2 CSV.GZ del mes (parte1 + parte2)
#   4. Extraer maestros (desde Archivos_de_apoyo.zip o directo)
# ══════════════════════════════════════════════════════════════════
import zipfile
from pathlib import Path

def _zip_name_para_mes(year, month, zips_disponibles):
    """Determina el nombre del ZIP que contiene el mes dado."""
    # Intentar año+semestre primero (ej: 2021A, 2021B)
    sem = 'A' if month <= 6 else 'B'
    if f'{year}{sem}.zip' in zips_disponibles:
        return f'{year}{sem}.zip'
    # Fallback: ZIP anual
    if f'{year}.zip' in zips_disponibles:
        return f'{year}.zip'
    return None

def _extraer_mes_de_zip(zip_local, mes_mmaaaa, dir_datos):
    """Extrae los 2 archivos CSV.GZ del mes desde el ZIP. Retorna lista de paths extraídos."""
    targets = [
        f'{mes_mmaaaa}_pais_parte1COMPLETO.csv.gz',
        f'{mes_mmaaaa}_pais_parte2COMPLETO.csv.gz',
    ]
    extraidos = []
    with zipfile.ZipFile(zip_local, 'r') as z:
        all_names = z.namelist()
        csvgz_en_zip = [n for n in all_names if '.csv.gz' in n]
        print(f'    Archivos CSV.GZ en ZIP: {len(csvgz_en_zip)}')
        if csvgz_en_zip:
            print(f'    Muestra: {sorted(csvgz_en_zip)[:4]}')

        for target in targets:
            dest = Path(dir_datos) / target
            if dest.exists() and dest.stat().st_size > 100_000:
                print(f'    ↩ {target} ya extraído ({dest.stat().st_size/1e6:.0f} MB)')
                extraidos.append(dest)
                continue
            # Buscar con o sin prefijo de carpeta, con o sin "COMPLETO"
            candidatos = [
                n for n in all_names
                if n.endswith(target) or n.endswith(target.replace('COMPLETO.csv.gz', '.csv.gz'))
            ]
            if candidatos:
                data = z.read(candidatos[0])
                with open(dest, 'wb') as f:
                    f.write(data)
                print(f'    ✓ {target}: {dest.stat().st_size/1e6:.0f} MB')
                extraidos.append(dest)
            else:
                print(f'    ⚠ No encontrado: {target}')
    return extraidos

def _descargar_drive(drive, file_id, output_path, label):
    from googleapiclient.http import MediaIoBaseDownload
    p = Path(output_path)
    if p.exists() and p.stat().st_size > 100_000:
        print(f'  ↩ {label} ya descargado ({p.stat().st_size/1e6:.0f} MB)')
        return
    print(f'  ↓ {label}...')
    req = drive.files().get_media(fileId=file_id)
    with open(output_path, 'wb') as f:
        dl = MediaIoBaseDownload(f, req, chunksize=100*1024*1024)
        done = False
        while not done:
            status, done = dl.next_chunk()
            if status:
                print(f'    {int(status.progress()*100)}%', end='\r')
    print(f'  ✓ {label}: {Path(output_path).stat().st_size/1e6:.0f} MB')

# ── Helper: extraer maestros de Archivos_de_apoyo.zip ─────────────
def _extraer_maestros_de_apoyo(apoyo_local, dir_datos, nombres):
    with zipfile.ZipFile(apoyo_local, 'r') as z:
        all_names = z.namelist()
        for nombre in nombres:
            dest = Path(dir_datos) / nombre
            if dest.exists() and dest.stat().st_size > 10_000:
                print(f'  ↩ {nombre} ya disponible')
                continue
            match = [n for n in all_names if nombre in n]
            if match:
                data = z.read(match[0])
                with open(dest, 'wb') as f:
                    f.write(data)
                print(f'  ✓ {nombre} (desde Archivos_de_apoyo.zip)')
            else:
                print(f'  ⚠ {nombre} no encontrado en Archivos_de_apoyo.zip')
                print(f'    Contenido del ZIP: {all_names}')

# ────────────────────────────────────────────────────────────────────
# 3a: manifest con IDs → gdown (sin autenticación)
# ────────────────────────────────────────────────────────────────────
if USAR_MANIFEST:
    import gdown

    year  = int(MES_ANALIZAR[:4])
    month = int(MES_ANALIZAR[5:7])
    mm, aaaa = f'{month:02d}', str(year)
    MES_MMAAAA = f'{mm}{aaaa}'

    # Encontrar el ZIP del manifest que contiene este mes
    zip_info_manifest = None
    for zip_nombre, info in zips_manifest.items():
        if MES_ANALIZAR in info.get('meses', []):
            zip_info_manifest = (zip_nombre, info['id'])
            break

    if zip_info_manifest is None:
        raise RuntimeError(f'{MES_ANALIZAR} no está en ningún ZIP del manifest.')

    zip_nombre, zip_id = zip_info_manifest
    zip_local = Path(f'{DIR_DATOS}/{zip_nombre}')
    print(f'  ZIP a descargar: {zip_nombre}')

    if not (zip_local.exists() and zip_local.stat().st_size > 1_000_000):
        print(f'  ↓ {zip_nombre}...')
        gdown.download(id=zip_id, output=str(zip_local), quiet=False, fuzzy=True)
    else:
        print(f'  ↩ {zip_nombre} ya descargado ({zip_local.stat().st_size/1e9:.1f} GB)')

    print(f'  Extrayendo archivos de {MES_ANALIZAR}...')
    _extraer_mes_de_zip(zip_local, MES_MMAAAA, DIR_DATOS)

    # Maestros
    datos_maestros = manifest.get('maestros', {})
    for nombre, id_key in [(NOMBRE_MAESTRO_PROD, 'productos_id'),
                           (NOMBRE_MAESTRO_SUC,  'sucursales_id')]:
        dest = Path(DIR_DATOS) / nombre
        if dest.exists() and dest.stat().st_size > 10_000:
            print(f'  ↩ {nombre} ya disponible')
            continue
        fid = datos_maestros.get(id_key, 'COMPLETAR')
        if fid not in ('COMPLETAR', '', None):
            print(f'  ↓ {nombre}...')
            gdown.download(id=fid, output=str(dest), quiet=False, fuzzy=True)
        elif datos_maestros.get('apoyo_zip_id', 'COMPLETAR') not in ('COMPLETAR', '', None):
            apoyo_local = Path(f'{DIR_DATOS}/Archivos_de_apoyo.zip')
            if not apoyo_local.exists():
                gdown.download(id=datos_maestros['apoyo_zip_id'],
                               output=str(apoyo_local), quiet=False, fuzzy=True)
            _extraer_maestros_de_apoyo(apoyo_local, DIR_DATOS, [nombre])

# ────────────────────────────────────────────────────────────────────
# 3b: sin manifest → Drive API con autenticación
# ────────────────────────────────────────────────────────────────────
else:
    from google.colab import auth
    from googleapiclient.discovery import build
    import io

    print('Autorizando acceso a Google Drive...')
    auth.authenticate_user()
    drive = build('drive', 'v3', cache_discovery=False)
    print('✓ Autorizado')

    def listar_directorio(folder_id):
        archivos, page_token = [], None
        while True:
            r = drive.files().list(
                q=f"'{folder_id}' in parents and trashed=false",
                fields='nextPageToken,files(id,name,size,mimeType)',
                pageSize=1000, pageToken=page_token
            ).execute()
            archivos.extend(r.get('files', []))
            page_token = r.get('nextPageToken')
            if not page_token:
                break
        return archivos

    print('Listando archivos en Drive...')
    todos = {}
    for fid in (FOLDER_HISTORICO, FOLDER_RECIENTE):
        for a in listar_directorio(fid):
            todos[a['name']] = a

    zips_disponibles = {n for n in todos if n.endswith('.zip')}
    print(f'  ZIPs encontrados: {sorted(zips_disponibles)}')

    # Determinar mes a analizar
    if not MES_ANALIZAR:
        from datetime import date
        hoy = date.today()
        mes_u = hoy.month - 1 if hoy.month > 1 else 12
        año_u = hoy.year if hoy.month > 1 else hoy.year - 1
        MES_ANALIZAR = f'{año_u}-{mes_u:02d}'
        print(f'  MES no especificado → usando último mes completo estimado: {MES_ANALIZAR}')

    year  = int(MES_ANALIZAR[:4])
    month = int(MES_ANALIZAR[5:7])
    mm, aaaa = f'{month:02d}', str(year)
    MES_MMAAAA = f'{mm}{aaaa}'

    # Mapear mes → ZIP
    zip_target = _zip_name_para_mes(year, month, zips_disponibles)
    if zip_target is None:
        raise RuntimeError(
            f'No se encontró ZIP para {MES_ANALIZAR}.\n'
            f'ZIPs disponibles en Drive: {sorted(zips_disponibles)}'
        )

    print(f'\n  Mes seleccionado: {MES_ANALIZAR}')
    print(f'  ZIP a descargar: {zip_target} ({int(todos[zip_target].get("size",0))/1e9:.1f} GB)')

    zip_local = Path(f'{DIR_DATOS}/{zip_target}')
    _descargar_drive(drive, todos[zip_target]['id'], zip_local, zip_target)

    print(f'\n  Extrayendo archivos de {MES_ANALIZAR}...')
    _extraer_mes_de_zip(zip_local, MES_MMAAAA, DIR_DATOS)

    # Maestros: buscar en Drive o en Archivos_de_apoyo.zip
    for nombre in (NOMBRE_MAESTRO_PROD, NOMBRE_MAESTRO_SUC):
        dest = Path(DIR_DATOS) / nombre
        if dest.exists() and dest.stat().st_size > 10_000:
            print(f'  ↩ {nombre} ya disponible')
            continue
        if nombre in todos:
            _descargar_drive(drive, todos[nombre]['id'], dest, nombre)
        elif 'Archivos_de_apoyo.zip' in todos:
            apoyo_local = Path(f'{DIR_DATOS}/Archivos_de_apoyo.zip')
            if not apoyo_local.exists():
                _descargar_drive(drive, todos['Archivos_de_apoyo.zip']['id'],
                                 apoyo_local, 'Archivos_de_apoyo.zip')
            _extraer_maestros_de_apoyo(apoyo_local, DIR_DATOS, [nombre])
        else:
            print(f'  ⚠ {nombre} no encontrado en Drive. Subilo manualmente a {DIR_DATOS}/')

    # Mostrar IDs descubiertos para poblar el manifest
    print('\n' + '='*60)
    print('IDs de ZIPs — para poblar data/drive_manifest.json:')
    print('='*60)
    zips_info = {}
    for zn, info in todos.items():
        if zn.endswith('.zip') and not zn.startswith('Archivos'):
            zips_info[zn] = info['id']
    print(json.dumps({
        'zips': {zn: {'id': fid, 'meses': ['COMPLETAR']} for zn, fid in sorted(zips_info.items())},
        'maestros': {
            'apoyo_zip_id': todos.get('Archivos_de_apoyo.zip', {}).get('id', 'no encontrado'),
            'productos_id': todos.get(NOMBRE_MAESTRO_PROD, {}).get('id', 'no encontrado'),
            'sucursales_id': todos.get(NOMBRE_MAESTRO_SUC, {}).get('id', 'no encontrado'),
        }
    }, indent=2))
    print('='*60)

print(f'\n✓ Archivos listos en {DIR_DATOS}/')

In [ ]:
# ══════════════════════════════════════════════════════════════════
# PASO 4 — Cargar y limpiar datos SEPA
# ══════════════════════════════════════════════════════════════════
import gc
import pandas as pd
from pathlib import Path

from sepa.pipeline.loader import iter_semester_csvgz, load_master_branches
from sepa.pipeline.cleaner import clean_prices
from sepa.pipeline.enricher import enrich_with_branches, filter_excluded_chains
from sepa.config.canasta import CANASTA_EANS

print(f'Cargando precios de {MES_ANALIZAR}...')
frames = []
for chunk in iter_semester_csvgz(Path(DIR_DATOS), ean_filter=CANASTA_EANS):
    chunk = filter_excluded_chains(chunk)
    chunk = clean_prices(chunk, auto_scale=True)
    frames.append(chunk)
    gc.collect()

if not frames:
    raise RuntimeError('Sin datos. Verificá que los CSV.GZ se descargaron correctamente.')

df_long = pd.concat(frames, ignore_index=True)
del frames; gc.collect()

print(f'✓ {len(df_long):,} registros | {df_long["ean_norm"].nunique()} EANs de canasta')
print(f'  Período: {df_long["fecha"].min().date()} → {df_long["fecha"].max().date()}')

In [ ]:
# ══════════════════════════════════════════════════════════════════
# PASO 5 — Enriquecer con maestros
# ══════════════════════════════════════════════════════════════════
mb = load_master_branches(f'{DIR_DATOS}/{NOMBRE_MAESTRO_SUC}')
df_enriched = enrich_with_branches(df_long, mb)
del df_long; gc.collect()

branch_cols = [c for c in ['id_comercio','id_bandera','id_sucursal'] if c in df_enriched.columns]
print(f'✓ {len(df_enriched[branch_cols].drop_duplicates()):,} sucursales | '
      f'{df_enriched.get("provincia", pd.Series(dtype=str)).nunique()} provincias | '
      f'{df_enriched.get("cadena", pd.Series(dtype=str)).nunique()} cadenas')

In [ ]:
# ══════════════════════════════════════════════════════════════════
# PASO 6 — Canasta mensual por sucursal
# ══════════════════════════════════════════════════════════════════
from sepa.pipeline.aggregator import compute_monthly_avg, build_branch_basket

df_monthly = compute_monthly_avg(df_enriched)
df_branch  = build_branch_basket(df_monthly)

df_mes = df_branch[df_branch['mes'] == MES_ANALIZAR].copy()
if df_mes.empty:
    raise RuntimeError(f'Sin datos para {MES_ANALIZAR}. Meses en los datos: {sorted(df_branch["mes"].unique())}')

avg = df_mes['canasta_total'].mean()
cv  = df_mes['canasta_total'].std() / avg * 100
print(f'══ ICM-UADE — {MES_ANALIZAR} ══')
print(f'Sucursales:       {len(df_mes):>8,}')
print(f'Canasta promedio: ${avg:>12,.0f}'.replace(',','.'))
print(f'Canasta mediana:  ${df_mes["canasta_total"].median():>12,.0f}'.replace(',','.'))
print(f'Dispersión (CV):  {cv:>11.1f}%')

In [ ]:
# ══════════════════════════════════════════════════════════════════
# PASO 7 — Mapa interactivo
# ══════════════════════════════════════════════════════════════════
from sepa.viz.maps import make_branch_map
from IPython.display import IFrame

suc_cols = branch_cols + [c for c in ['lat','lng','cadena','provincia','region',
                                       'sucursales_nombre','localidad_nombre','barrio_caba']
                           if c in df_enriched.columns]
df_suc = df_enriched[suc_cols].drop_duplicates(subset=branch_cols)

mapa_path = f'/content/mapa_canasta_pais_{MES_MMAAAA}_filtros.html'
make_branch_map(df_mes, df_suc, mapa_path, mes=MES_ANALIZAR)
print(f'✓ Mapa generado ({Path(mapa_path).stat().st_size/1e6:.0f} MB)')
IFrame(mapa_path, width='100%', height=600)

In [ ]:
# ══════════════════════════════════════════════════════════════════
# PASO 8 — Rankings de cadenas
# ══════════════════════════════════════════════════════════════════
from sepa.analysis.chains import national_ranking, amba_ranking
from sepa.viz.charts import plot_chain_ranking
from IPython.display import Image, display as ipy_display

rank_nac  = national_ranking(df_mes, df_suc, mes=MES_ANALIZAR)
rank_amba = amba_ranking(df_mes, df_suc, mes=MES_ANALIZAR)

print('── Ranking Nacional ──')
display(rank_nac[['ranking','cadena','canasta_promedio','n_sucursales','vs_promedio_pct']]
        .style.format({'canasta_promedio':'${:,.0f}','vs_promedio_pct':'{:+.1f}%'}).hide(axis='index'))

png_nac = f'/content/ranking_cadenas_nacional_{MES_MMAAAA}.png'
plot_chain_ranking(rank_nac, png_nac, title=f'Ranking Nacional — {MES_ANALIZAR}')
ipy_display(Image(png_nac))

if not rank_amba.empty:
    print('\n── Ranking AMBA ──')
    display(rank_amba[['ranking','cadena','canasta_promedio','n_sucursales','vs_promedio_pct']]
            .style.format({'canasta_promedio':'${:,.0f}','vs_promedio_pct':'{:+.1f}%'}).hide(axis='index'))
    png_amba = f'/content/ranking_cadenas_amba_{MES_MMAAAA}.png'
    plot_chain_ranking(rank_amba, png_amba, title=f'Ranking AMBA — {MES_ANALIZAR}')
    ipy_display(Image(png_amba))

In [ ]:
# ══════════════════════════════════════════════════════════════════
# PASO 9 — Ranking provincial
# ══════════════════════════════════════════════════════════════════
from sepa.pipeline.aggregator import aggregate_by_province
from sepa.analysis.basket import basket_by_province
from sepa.viz.charts import plot_province_ranking

if 'provincia' in df_suc.columns:
    df_prov = aggregate_by_province(df_mes, df_suc)
    df_rank_prov = basket_by_province(df_prov, MES_ANALIZAR)
    print('── Ranking Provincial ──')
    display(df_rank_prov[['ranking','provincia','canasta_provincia','vs_promedio_pct']]
            .style.format({'canasta_provincia':'${:,.0f}','vs_promedio_pct':'{:+.1f}%'}).hide(axis='index'))
    png_prov = f'/content/ranking_provincias_{MES_MMAAAA}.png'
    plot_province_ranking(df_prov, png_prov, mes=MES_ANALIZAR)
    ipy_display(Image(png_prov))

In [ ]:
# ══════════════════════════════════════════════════════════════════
# PASO 10 — Ranking barrios CABA
# ══════════════════════════════════════════════════════════════════
from sepa.analysis.basket import barrio_ranking_caba

df_barrios = barrio_ranking_caba(df_mes, df_suc, mes=MES_ANALIZAR)
if not df_barrios.empty:
    print(f'── Ranking Barrios CABA ({len(df_barrios)} barrios) ──')
    display(df_barrios[['ranking','barrio_caba','canasta_barrio','n_sucursales','vs_promedio_caba_pct']]
            .rename(columns={'barrio_caba':'barrio','canasta_barrio':'canasta','vs_promedio_caba_pct':'vs CABA %'})
            .style.format({'canasta':'${:,.0f}','vs CABA %':'{:+.1f}%'}).hide(axis='index'))

In [ ]:
# ══════════════════════════════════════════════════════════════════
# PASO 11 — Descargar outputs a tu computadora
# ══════════════════════════════════════════════════════════════════
from google.colab import files

outputs = [
    (f'/content/mapa_canasta_pais_{MES_MMAAAA}_filtros.html', 'mapa HTML interactivo'),
    (f'/content/ranking_cadenas_nacional_{MES_MMAAAA}.png',   'ranking nacional PNG'),
    (f'/content/ranking_cadenas_amba_{MES_MMAAAA}.png',       'ranking AMBA PNG'),
    (f'/content/ranking_provincias_{MES_MMAAAA}.png',          'ranking provincial PNG'),
]

print(f'Descargando outputs de {MES_ANALIZAR} a tu computadora...')
for ruta, label in outputs:
    if Path(ruta).exists():
        files.download(ruta)
        print(f'  ✓ {label}')
    else:
        print(f'  ⚠ No generado: {label}')

print(f'\n✓ Análisis de {MES_ANALIZAR} completado.')